<div style="background:linear-gradient(120deg,#0b3a67 0%,#0f6cbd 60%,#2aa0a4 100%);background-color:#0f6cbd;border-radius:12px;padding:22px 28px;color:#ffffff"><table style="border:none;border-collapse:collapse;width:100%;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0;vertical-align:middle;text-align:left;color:#ffffff"><div style="font-size:13px;letter-spacing:2px;text-transform:uppercase;opacity:0.85">Microsoft Fabric &middot; Lakehouse</div><div style="font-size:34px;font-weight:700;line-height:1.2;margin:2px 0 6px 0">Backup Utility</div><div style="font-size:15px;line-height:1.5;max-width:520px">Takes a full, checked copy of a lakehouse (Delta tables and Files) and stores it in a write-once ADLS Gen2 container, where it cannot be changed or deleted until the retention period ends.</div></td><td style="border:none;padding:0 0 0 20px;vertical-align:middle;text-align:right"></td></tr></table></div><div style="margin:12px 2px 0 2px;font-size:14px;line-height:1.6"><b>How to use this notebook:</b> run the cells from top to bottom, <b>one at a time</b>. Do not use <i>Run all</i>: the backup must wait until the form is filled in.<br/><span style="display:inline-block;background:#107c10;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Steps 1 to 2 change nothing</span> &nbsp; <span style="display:inline-block;background:#c4314b;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Step 3 writes the backup</span> &nbsp; <div style="margin:10px 2px 0 2px;font-size:13px;line-height:1.55;opacity:0.9"><b>Before you start:</b> make sure nothing is writing to the source lakehouse. Each run is a full copy that cannot be removed early. Not included: shortcuts, permissions, security roles, SQL endpoint views.</div>


<div style="border-left:6px solid #0f6cbd;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#0f6cbd;font-weight:700">Step 1</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Load the backup functions</div><div style="font-size:14px;line-height:1.55">Loads the functions that do the copying and checking from <code>nb_backup_restore_adls_v6</code>. It copies nothing.<br/><b>Expect:</b> the cell finishes in a few seconds with no output. <b>If it fails:</b> that notebook is missing from this workspace.</div></td></tr></table></div>


In [ ]:
%run nb_backup_restore_adls_v6

<div style="border-left:6px solid #5c2e91;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#5c2e91;font-weight:700">Step 2</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Prepare the form</div><div style="font-size:14px;line-height:1.55">Defines the input form. Nothing to read or edit here; the cell can stay collapsed.<br/><b>Expect:</b> no output.</div></td></tr></table></div>


In [ ]:
%run nb_backup_form

<div style="border-left:6px solid #c4314b;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#c4314b;font-weight:700">Step 3</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Run the backup</div><div style="font-size:14px;line-height:1.55"><span style="display:inline-block;background:#c4314b;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Writes data that cannot be deleted</span><br/>Copies every table, and Files if ticked, to the archive, then writes the manifest that marks the backup complete. A lakehouse of about 50 tables takes several minutes.<br/><b>Expect:</b> one line per table, ending with <code>BACKUP COMPLETE: &lt;timestamp&gt; | N tables | ...</code>. Note the timestamp: it identifies this backup for a restore.<br/><b>If it ends with</b> <code>BACKUP FAILED</code>: read the reason on that line, fix it and run this cell again. The unfinished run stays in the archive and is ignored by restore.</div></td></tr></table></div>


In [ ]:
p = form.params
if p is None:
    raise RuntimeError("Press Get IDs on the form first.")

ARCHIVE = p["backup_container"]

BACKUP_TS = backup_lakehouse_to_archive(
    p["source_workspace_id"], p["source_lakehouse_id"], ARCHIVE,
    staging_ws_id=p["staging_workspace_id"], staging_lh_id=p["staging_lakehouse_id"],
    copy_files=p["copy_files"],
    verify_counts=p["verify_counts"],
)